# Concept-only Action Classification (No CRL)

This notebook tests whether the 72-dimensional concept representation is sufficient for action recognition without the CRL reasoner.

The runnable synthetic experiment estimates a controlled upper-bound-like result from class concept prototypes. The real-data section evaluates cached concept logits when `concept_logits.npz` is available. It never imports or uses CRL.

## Experiment design

Input: concept probabilities $p n [0,1]^{72}$.

Classifier: a linear layer and a small MLP trained directly on concepts.

Reported metrics:
- concept micro/macro F1, when concept targets are available;
- action Top-1 and Top-5 accuracy;
- the gap between linear and MLP action heads.

Important: synthetic accuracy is not NTU accuracy. It only checks whether the concept-to-action mapping is learnable under controlled noise.

In [ ]:
from pathlib import Path
import csv
import os
import sys
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Colab setup: clone/upload the repository to /content/CLIR-SkateFormer,
# or set USE_GOOGLE_DRIVE=True and place the project under MyDrive/Thesis.
IN_COLAB = 'google.colab' in sys.modules
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/Thesis') if IN_COLAB and USE_GOOGLE_DRIVE else None,
    Path('/content'),
    Path(r'D:/Thesis'),
]
ROOT = next((path for path in CANDIDATE_ROOTS if path is not None and (path / 'CLIR-SkateFormer').exists()), Path('/content') if IN_COLAB else Path(r'D:/Thesis'))
PROJECT_ROOT = ROOT / 'CLIR-SkateFormer'
CONCEPT_CSV = PROJECT_ROOT / 'concepts' / 'ntu_rgbd_60_120.csv'
CACHE_PATH = ROOT / 'concept_logits.npz'
NUM_CONCEPTS = 72
print('device:', DEVICE)
print('project root:', PROJECT_ROOT)
print('concept csv:', CONCEPT_CSV)
print('optional cache:', CACHE_PATH)
if not CONCEPT_CSV.exists():
    print('Synthetic mode is ready. For real mode, clone/upload CLIR-SkateFormer under /content or mount Drive.')

In [ ]:
def load_action_concepts(path):
    with path.open('r', newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    if not rows or 'action_class' not in rows[0]:
        raise ValueError(f'Invalid concept CSV: {path}')
    names = [name for name in rows[0] if name != 'action_class']
    if len(names) == 74:
        names = [name for i, name in enumerate(names) if i not in (50, 65)]
    elif len(names) > 72:
        names = [name for name in names if name not in ('foot_tap', 'temporal_sequence_seq_cascading')]
    matrix = np.asarray([[float(row[name]) for name in names] for row in rows], dtype=np.float32)
    if matrix.shape[1] != NUM_CONCEPTS:
        raise ValueError(f'Expected 72 concepts, got {matrix.shape}')
    return matrix, names

class_concepts, concept_names = load_action_concepts(CONCEPT_CSV)
NUM_CLASSES = class_concepts.shape[0]
print('classes:', NUM_CLASSES, 'concepts:', class_concepts.shape[1])
print('mean positive concepts per class:', class_concepts.sum(axis=1).mean())
print('duplicate prototype rows:', NUM_CLASSES - np.unique(class_concepts, axis=0).shape[0])

## 1. Synthetic sanity experiment

Each class uses its CSV concept prototype. We perturb the prototype with logit noise to simulate imperfect concept predictions, then train action heads using only concept probabilities.

In [ ]:
def make_synthetic_split(prototypes, samples_per_class, noise_std, seed):
    generator = torch.Generator().manual_seed(seed)
    targets = torch.from_numpy(np.repeat(prototypes, samples_per_class, axis=0)).float()
    labels = torch.arange(prototypes.shape[0]).repeat_interleave(samples_per_class)
    clean_logits = torch.logit(targets.clamp(0.02, 0.98))
    noisy_logits = clean_logits + noise_std * torch.randn(clean_logits.shape, generator=generator)
    probabilities = noisy_logits.sigmoid()
    return probabilities, labels, targets

train_x, train_y, train_targets = make_synthetic_split(class_concepts, 24, 0.80, SEED)
test_x, test_y, test_targets = make_synthetic_split(class_concepts, 8, 0.80, SEED + 1)
print('train:', tuple(train_x.shape), 'test:', tuple(test_x.shape))

In [ ]:
def multilabel_metrics(probabilities, targets, threshold=0.5):
    predictions = probabilities >= threshold
    truth = targets >= threshold
    tp = (predictions & truth).sum(0).float()
    fp = (predictions & ~truth).sum(0).float()
    fn = (~predictions & truth).sum(0).float()
    precision = tp / (tp + fp).clamp_min(1.0)
    recall = tp / (tp + fn).clamp_min(1.0)
    f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-8)
    micro_f1 = 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum()).clamp_min(1.0)
    macro_f1 = f1.mean()
    return {'micro_f1': float(micro_f1), 'macro_f1': float(macro_f1)}

print('synthetic concept metrics:', multilabel_metrics(test_x, test_targets))

In [ ]:
class ActionHead(nn.Module):
    def __init__(self, kind, num_concepts, num_classes):
        super().__init__()
        if kind == 'linear':
            self.net = nn.Linear(num_concepts, num_classes)
        elif kind == 'mlp':
            self.net = nn.Sequential(
                nn.Linear(num_concepts, 128), nn.GELU(), nn.Dropout(0.10),
                nn.Linear(128, num_classes)
            )
        else:
            raise ValueError(kind)
    def forward(self, x):
        return self.net(x)

def accuracy(logits, labels):
    top1 = (logits.argmax(1) == labels).float().mean()
    top5 = logits.topk(min(5, logits.shape[1]), 1).indices.eq(labels[:, None]).any(1).float().mean()
    return float(top1), float(top5)

def train_head(kind, train_x, train_y, test_x, test_y, epochs=100):
    model = ActionHead(kind, train_x.shape[1], int(train_y.max()) + 1).to(DEVICE)
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=128, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    for _ in range(epochs):
        model.train()
        for x_batch, y_batch in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x_batch.to(DEVICE)), y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        logits = model(test_x.to(DEVICE)).cpu()
    return model, accuracy(logits, test_y), logits

for head_kind in ('linear', 'mlp'):
    _, scores, _ = train_head(head_kind, train_x, train_y, test_x, test_y)
    print(f'{head_kind:>6} | Top-1: {scores[0] * 100:.2f}% | Top-5: {scores[1] * 100:.2f}%')

## 2. Real-data mode: train from scratch without CRL

This section trains a separate concept-only baseline from random initialization. It does not load a CLIR checkpoint and does not instantiate or call CRL. The optional cache contains:

- `train_concept_logits`, `test_concept_logits`: shape `[N, 72]`;
- `train_labels`, `test_labels`: shape `[N]`;
- optional `train_concept_targets`, `test_concept_targets`: shape `[N, 72]`.

The logits are converted with sigmoid. This section does not call the CRL reasoner.

## 3. Train the real-data concept-only model

The model is `SkateFormer backbone -> concept heads -> action MLP`. Both action loss and concept BCE are used during training.

In [ ]:
import gc
import yaml
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DATA_CANDIDATES = [
    ROOT / 'data' / 'ntu' / 'NTU60_CS.npz',
    PROJECT_ROOT / 'data' / 'ntu' / 'NTU60_CS.npz',
    Path('/content/drive/MyDrive/Thesis/data/ntu/NTU60_CS.npz'),
    Path('/content/data/ntu/NTU60_CS.npz'),
]
DATA_NPZ = next((p for p in DATA_CANDIDATES if p.exists()), ROOT / 'data' / 'ntu' / 'NTU60_CS.npz')

CONFIG_YAML = PROJECT_ROOT / 'config' / 'train' / 'ntu60_cs' / 'CLIR_SkateFormer_j.yaml'
OUTPUT_CACHE = ROOT / 'concept_logits.npz'
BASELINE_CHECKPOINT = ROOT / 'concept_only_no_crl.pt'
EPOCHS = 30
BATCH_SIZE = 32

BACKBONE_CANDIDATES = [
    Path('/content/drive/MyDrive/Thesis/pretrained/ntu60_CSub/SkateFormer_j.pt') if IN_COLAB and USE_GOOGLE_DRIVE else None,
    Path('/content/pretrained/ntu60_CSub/SkateFormer_j.pt') if IN_COLAB else None,
    PROJECT_ROOT / 'pretrained' / 'ntu60_CSub' / 'SkateFormer_j.pt',
    ROOT / 'pretrained' / 'ntu60_CSub' / 'SkateFormer_j.pt',
]
BACKBONE_WEIGHTS = next((path for path in BACKBONE_CANDIDATES if path is not None and path.exists()), None)

if not DATA_NPZ.exists():
    raise FileNotFoundError(f'Missing NTU data: {DATA_NPZ}')
print('data path:', DATA_NPZ)
print('pretrained backbone:', BACKBONE_WEIGHTS if BACKBONE_WEIGHTS else 'not found; random initialization')

sys.path.insert(0, str(PROJECT_ROOT))
from feeders.feeder_ntu import Feeder
from model.SkateFormer import SkateFormer_

config = yaml.safe_load(CONFIG_YAML.read_text(encoding='utf-8'))
feeder_args = {}
for split in ('train', 'test'):
    feeder_args[split] = dict(config[f'{split}_feeder_args'])
    feeder_args[split]['data_path'] = str(DATA_NPZ)
    feeder_args[split]['concepts_csv'] = str(CONCEPT_CSV)
    feeder_args[split]['split'] = split
    feeder_args[split]['debug'] = False

# Load feeders
print('Loading train feeder...')
train_feeder = Feeder(**feeder_args['train'])
# Optimize RAM: convert float32 (7.2GB) to float16 (3.6GB)
train_feeder.data = train_feeder.data.astype(np.float16)

print('Loading test feeder...')
test_feeder = Feeder(**feeder_args['test'])
# Optimize RAM: convert float32 (3.0GB) to float16 (1.5GB)
test_feeder.data = test_feeder.data.astype(np.float16)

gc.collect()

# CRITICAL FIX FOR COLAB RAM:
# 1. num_workers=0 avoids multiprocessing fork and Linux Copy-on-Write page duplication
# 2. pin_memory=False prevents page-locked host memory allocation failures when RAM is near capacity
train_loader = DataLoader(train_feeder, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_feeder, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
NUM_CLASSES_REAL = int(max(train_feeder.label.max(), test_feeder.label.max()) + 1)

class ConceptOnlyModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        args = config['model_args']
        self.backbone = SkateFormer_(
            num_classes=num_classes,
            num_people=args['num_people'],
            num_points=args['num_points'],
            num_frames=args['num_frames'],
            num_heads=args['num_heads'],
            type_1_size=tuple(args['type_1_size']),
            type_2_size=tuple(args['type_2_size']),
            type_3_size=tuple(args['type_3_size']),
            type_4_size=tuple(args['type_4_size']),
            attn_drop=args['attn_drop'],
            head_drop=args['head_drop'],
            rel=args['rel'],
            drop_path=args['drop_path'],
            mlp_ratio=args['mlp_ratio'],
            index_t=args['index_t'],
        )
        feature_dim = self.backbone.head.in_features
        self.spatial_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.temporal_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.spatial_head = nn.Sequential(nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 52))
        self.temporal_head = nn.Sequential(nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 20))
        self.action_head = nn.Sequential(nn.Linear(NUM_CONCEPTS, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, num_classes))

    def forward(self, data, index_t):
        feature_map = self.backbone.forward_features_from_input(data, index_t=index_t)
        spatial_tokens = feature_map.mean(dim=2)
        spatial_attn = torch.softmax(self.spatial_pool_score(spatial_tokens), dim=-1)
        spatial_feat = (spatial_tokens * spatial_attn).sum(dim=-1)
        temporal_tokens = feature_map.mean(dim=3)
        temporal_attn = torch.softmax(self.temporal_pool_score(temporal_tokens), dim=-1)
        temporal_feat = (temporal_tokens * temporal_attn).sum(dim=-1)
        concept_logits = torch.cat([self.spatial_head(spatial_feat), self.temporal_head(temporal_feat)], dim=-1)
        action_logits = self.action_head(concept_logits.sigmoid())
        return action_logits, concept_logits

model = ConceptOnlyModel(NUM_CLASSES_REAL).to(DEVICE)
if BACKBONE_WEIGHTS is not None:
    raw_checkpoint = torch.load(BACKBONE_WEIGHTS, map_location='cpu')
    if isinstance(raw_checkpoint, dict):
        raw_checkpoint = raw_checkpoint.get('state_dict', raw_checkpoint.get('model', raw_checkpoint))
    backbone_state = model.backbone.state_dict()
    compatible = {}
    for key, value in raw_checkpoint.items():
        clean_key = key.replace('module.', '', 1)
        if clean_key.startswith('backbone.'):
            clean_key = clean_key[len('backbone.'):]
        if clean_key in backbone_state and backbone_state[clean_key].shape == value.shape:
            compatible[clean_key] = value
    missing, unexpected = model.backbone.load_state_dict(compatible, strict=False)
    print(f'Loaded pretrained backbone tensors: {len(compatible)}; missing: {len(missing)}')

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.05)
concept_loss_fn = nn.BCEWithLogitsLoss()
best_top1 = -1.0

def run_epoch(loader, training, collect_outputs=False, desc=None):
    model.train(training)
    total_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total_samples = 0
    all_logits, all_labels, all_concepts, all_targets = [], [], [], []
    
    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    for data, index_t, label, _, concept_target in pbar:
        data = data.float().to(DEVICE, non_blocking=True)
        index_t = index_t.float().to(DEVICE, non_blocking=True)
        label = label.long().to(DEVICE, non_blocking=True)
        concept_target = concept_target.float().to(DEVICE, non_blocking=True)
        
        with torch.set_grad_enabled(training):
            action_logits, concept_logits = model(data, index_t)
            loss = nn.functional.cross_entropy(action_logits, label) + 0.5 * concept_loss_fn(concept_logits, concept_target)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
        bs = label.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        
        with torch.no_grad():
            correct_top1 += (action_logits.argmax(1) == label).sum().item()
            k = min(5, action_logits.shape[1])
            correct_top5 += action_logits.topk(k, 1).indices.eq(label[:, None]).any(1).sum().item()
            
        pbar.set_postfix({
            'loss': f'{total_loss / max(1, total_samples):.4f}',
            'top1': f'{correct_top1 / max(1, total_samples) * 100:.2f}%',
        })
            
        if collect_outputs:
            all_logits.append(action_logits.detach().cpu())
            all_labels.append(label.cpu())
            all_concepts.append(concept_logits.detach().cpu())
            all_targets.append(concept_target.cpu())
            
    top1 = correct_top1 / max(1, total_samples)
    top5 = correct_top5 / max(1, total_samples)
    avg_loss = total_loss / max(1, total_samples)
    
    if collect_outputs:
        logits = torch.cat(all_logits)
        labels = torch.cat(all_labels)
        raw_concept_logits = torch.cat(all_concepts)
        concepts = raw_concept_logits.sigmoid()
        targets = torch.cat(all_targets)
        concept_scores = multilabel_metrics(concepts, targets)
        return avg_loss, top1, top5, concept_scores, logits, labels, raw_concept_logits, targets
    else:
        return avg_loss, top1, top5, None, None, None, None, None

for epoch in range(1, EPOCHS + 1):
    train_stats = run_epoch(train_loader, training=True, collect_outputs=False, desc=f'Epoch {epoch:02d}/{EPOCHS:02d} [Train]')
    test_stats = run_epoch(test_loader, training=False, collect_outputs=True, desc=f'Epoch {epoch:02d}/{EPOCHS:02d} [Test]')
    if test_stats[1] > best_top1:
        best_top1 = test_stats[1]
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'top1': best_top1}, BASELINE_CHECKPOINT)
    print(
        f'Epoch {epoch:03d} | train loss {train_stats[0]:.4f} | train Top-1 {train_stats[1]*100:.2f}% | '
        f'test Top-1 {test_stats[1] * 100:.2f}% | Top-5 {test_stats[2] * 100:.2f}% | '
        f'concept F1 micro/macro {test_stats[3]["micro_f1"]:.4f}/{test_stats[3]["macro_f1"]:.4f}'
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Export the final model's concept logits for the no-CRL action-head comparison.
print('Exporting concept cache for downstream evaluation...')
model.eval()
train_stats = run_epoch(train_loader, training=False, collect_outputs=True, desc='Exporting Train')
test_stats = run_epoch(test_loader, training=False, collect_outputs=True, desc='Exporting Test')

np.savez_compressed(
    OUTPUT_CACHE,
    train_concept_logits=train_stats[6].numpy(),
    test_concept_logits=test_stats[6].numpy(),
    train_labels=train_stats[5].numpy(),
    test_labels=test_stats[5].numpy(),
    train_concept_targets=train_stats[7].numpy(),
    test_concept_targets=test_stats[7].numpy(),
)
print('Saved baseline checkpoint:', BASELINE_CHECKPOINT)
print('Saved concept cache:', OUTPUT_CACHE)


In [ ]:
def evaluate_real_cache(path):
    if not path.exists():
        print(f'No real-data cache found: {path}')
        print('Run the synthetic section now, then provide concept_logits.npz for NTU evaluation.')
        return
    cache = np.load(path)
    required = ('train_concept_logits', 'test_concept_logits', 'train_labels', 'test_labels')
    missing = [key for key in required if key not in cache]
    if missing:
        raise KeyError(f'Missing arrays in {path}: {missing}')
    real_train_x = torch.from_numpy(cache['train_concept_logits']).float().sigmoid()
    real_test_x = torch.from_numpy(cache['test_concept_logits']).float().sigmoid()
    real_train_y = torch.from_numpy(cache['train_labels']).long()
    real_test_y = torch.from_numpy(cache['test_labels']).long()
    if 'test_concept_targets' in cache:
        target_metrics = multilabel_metrics(real_test_x, torch.from_numpy(cache['test_concept_targets']).float())
        print('real concept metrics:', target_metrics)
    for head_kind in ('linear', 'mlp'):
        _, scores, _ = train_head(head_kind, real_train_x, real_train_y, real_test_x, real_test_y)
        print(f'real {head_kind:>6} | Top-1: {scores[0] * 100:.2f}% | Top-5: {scores[1] * 100:.2f}%')

evaluate_real_cache(CACHE_PATH)

## Interpretation

- High synthetic accuracy only shows that the CSV concept prototypes contain enough class information under the chosen noise level.
- Real-data Top-1 is the relevant result for the no-CRL baseline.
- Compare the linear head and MLP: a small gap suggests concepts are already linearly separable; a large gap suggests nonlinear concept interactions are needed.
- Compare predicted-concept inputs with ground-truth concept inputs when both are available to measure how much action accuracy is lost in concept prediction.